In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import sys

sys.path.append("..")  # Go back to base directory

from modules.graph import *
from modules.viewer3d import *

In [ ]:
def dh_transform(theta, d, a, alpha):
    """Compute individual DH transformation matrix."""
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)

    return np.array([
        [ct, -st * ca,  st * sa, a * ct],
        [st,  ct * ca, -ct * sa, a * st],
        [ 0,       sa,       ca,      d],
        [ 0,        0,        0,      1]
    ])

def forward_kinematics_all_frames(dh_params, joint_values):
    """
    Return a list of transformation matrices (one for each frame).
    """
    T = np.eye(4)
    frames = []  # base frame

    for i, (theta, d, a, alpha) in enumerate(dh_params):
        if theta == 'q':  # revolute
            current_theta = joint_values[i]
            current_d = d
        else:  # prismatic
            current_theta = theta
            current_d = joint_values[i]

        A_i = dh_transform(current_theta, current_d, a, alpha)
        T = T @ A_i
        frames.append(T.copy())

    return frames


In [ ]:
def dh_transform_torch(theta, d, a, alpha):
    """Compute DH transformation matrix (torch version, supports autograd)."""
    # ensure everything is a tensor
    theta = torch.as_tensor(theta, dtype=torch.float32, device=d.device if isinstance(d, torch.Tensor) else 'cpu')
    d     = torch.as_tensor(d, dtype=torch.float32, device=theta.device)
    a     = torch.as_tensor(a, dtype=torch.float32, device=theta.device)
    alpha = torch.as_tensor(alpha, dtype=torch.float32, device=theta.device)

    ct, st = torch.cos(theta), torch.sin(theta)
    ca, sa = torch.cos(alpha), torch.sin(alpha)

    T = torch.stack([
        torch.stack([ct, -st * ca,  st * sa, a * ct]),
        torch.stack([st,  ct * ca, -ct * sa, a * st]),
        torch.stack([torch.zeros((), device=theta.device, dtype=torch.float32), sa, ca, d]),
        torch.tensor([0., 0., 0., 1.], dtype=torch.float32, device=theta.device)
    ])
    
    return T


def forward_kinematics_all_frames_torch(dh_params, joint_values):
    """
    Compute forward kinematics for all frames (torch version).
    dh_params: list of (theta, d, a, alpha), where theta can be 'q' (revolute) or numeric,
               and d can be 'q' (prismatic) or numeric.
    joint_values: torch tensor of joint values (shape: [n_joints])
    """
    T = torch.eye(4, dtype=joint_values.dtype, device=joint_values.device)
    frames = []

    for i, (theta, d, a, alpha) in enumerate(dh_params):
        if theta == 'q':  # revolute
            current_theta = joint_values[i]
            current_d = torch.as_tensor(d, dtype=joint_values.dtype, device=joint_values.device)
        else:  # prismatic
            current_theta = torch.as_tensor(theta, dtype=joint_values.dtype, device=joint_values.device)
            current_d = joint_values[i]

        A_i = dh_transform_torch(current_theta, current_d, a, alpha)
        T = T @ A_i
        frames.append(T.clone())  # keep a copy

    return frames

In [ ]:
L1 = 0.4
L2 = 0.3
L3 = 0.2

dh_params = [
    ('q', 0.0, L1, 0.0),
    ('q', 0.0, L2, 0.0),
    ('q', 0.0, L3, 0.0)
]

joint_limits = [
    [-np.pi, np.pi],
    [0, 2 * np.pi/3],
    [0, 2 * np.pi/3],
]

In [ ]:
dataset_size = 100000

# Randomize based on joint limits
output_data = []
for low, high in joint_limits:
    random_joint_values = np.random.uniform(
        low=low,
        high=high,
        size=dataset_size,
    )

    output_data.append(random_joint_values)
output_data = np.array(output_data).T

# Full randomize
#output_data = np.random.uniform(low=0.0, high=np.pi, size=(dataset_size, len(dh_params)))

input_data = []
for j in output_data:
    input_data.append(forward_kinematics_all_frames(dh_params, j)[-1][0:3, [-1]].flatten()[:3])

input_data = np.array(input_data)

In [ ]:
class Model(nn.Module):
  def __init__(self, in_features=3, h1=200, h2=200, h3=200, out_features=len(dh_params)):
    super().__init__()

    # Neural network structure, like literal connections to neurons
    self.fc1 = nn.Linear(in_features, h1)
    self.fc2 = nn.Linear(h1, h2)
    self.fc3 = nn.Linear(h2, h3)
    self.out = nn.Linear(h3, out_features)

  # This is the computation function, put the input, get the output
  def forward(self, x):
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = F.relu(self.fc3(x))
    x = F.tanh(self.out(x)) * np.pi

    return x

In [ ]:
# BEST ONE YET !!

class Model(nn.Module):
  def __init__(self, in_features=3, h1=200, h2=200, h3=200, out_features=len(dh_params)):
    super().__init__()

    # Neural network structure, like literal connections to neurons
    self.fc1 = nn.Linear(in_features, h1)
    self.fc2 = nn.Linear(h1, h2)
    self.fc3 = nn.Linear(h2, h3)
    self.out = nn.Linear(h3, out_features)

  # This is the computation function, put the input, get the output
  def forward(self, x):
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = F.relu(self.fc3(x))
    x = F.tanh(self.out(x)) * np.pi

    return x

In [ ]:
'''# SIMPLE BUT WORKS

class Model(nn.Module):
  def __init__(self, in_features=3, h1=100, h2=100, h3=100, out_features=len(dh_params)):
    super().__init__()

    # Neural network structure, like literal connections to neurons
    self.fc1 = nn.Linear(in_features, h1)
    self.fc2 = nn.Linear(h1, h2)
    self.fc3 = nn.Linear(h2, h3)
    self.out = nn.Linear(h3, out_features)

  # This is the computation function, put the input, get the output
  def forward(self, x):
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = F.relu(self.fc3(x))
    x = self.out(x)

    return x'''

In [ ]:
torch.manual_seed = 42

model = Model()

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(input_data, output_data, test_size=0.2, random_state=torch.manual_seed)

x_train = torch.FloatTensor(x_train)
x_test = torch.FloatTensor(x_test)
y_train = torch.FloatTensor(y_train)
y_test = torch.FloatTensor(y_test)

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 110
losses = []

for i in range(epochs):
  # Use model
  y_pred = model.forward(x_train)

  # Compute loss function
  loss = criterion(y_pred, y_train)

  losses.append(loss.detach().numpy())

  # Only prints every 10 epochs
  if i % 10 == 0:
    print(f"Epoch: {i}, Loss: {loss}")

  optimizer.zero_grad() # Resets the gradient
  loss.backward() # Computes gradient of loss for each neuron
  optimizer.step() # Update weights and biases based on the gradient

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.plot(range(epochs), losses)
plt.ylabel("Error")
plt.xlabel("Epoch")

In [ ]:
from coppeliasim_zmqremoteapi_client import RemoteAPIClient

# Init client
client = RemoteAPIClient()  # Client object
sim = client.getObject("sim")  # Simulation object

In [ ]:
defaultIdleFps = sim.getInt32Param(sim.intparam_idle_fps)
sim.setInt32Param(sim.intparam_idle_fps, 0)

joint_handles = [
    sim.getObject("/J0"),
    sim.getObject("/J0/L1/J1"),
    sim.getObject("/J0/L1/J1/L2/J2")
]
goal_handle = sim.getObject("/Goal")

# Simulation begins here
sim.startSimulation()

# Retrieve images
while not sim.getSimulationStopping():
    goal_position = sim.getObjectPosition(goal_handle)

    with torch.no_grad():
        predicted_joints = model.forward(torch.FloatTensor(goal_position)).numpy().tolist()

    for j, a in zip(joint_handles, predicted_joints):
        sim.setJointPosition(j, a)

# Restore the original idle loop frequency:
sim.setInt32Param(sim.intparam_idle_fps, defaultIdleFps)

# Simulation ends here
sim.stopSimulation()